# California Housing Price Prediction: Full Pipeline

Technical notebook for the Technical Report (Appendix). Open this notebook from the project folder (the one that contains `src/` and `data/`) and choose **Run All**. Each step runs one script from `src/` in order. Install the packages first with `pip install -r requirements.txt` in a terminal.

## 0. Environment

In [1]:
import sys, pandas, numpy, sklearn, scipy
print('Python', sys.version.split()[0])
for m in (pandas, numpy, sklearn, scipy):
    print(m.__name__, m.__version__)

Python 3.11.15
pandas 3.0.2
numpy 2.4.4
sklearn 1.8.0
scipy 1.17.1


## 1. Data cleaning and preparation (`src/data_prep.py`)
Removes duplicates and invalid rows, drops top-coded targets, imputes missing bedrooms, engineers ratio features, removes IQR outliers, and saves `data/housing_clean.csv`.

In [2]:
%run src/data_prep.py

INFO: Loading raw data from <project>/data/housing.csv


INFO: Loaded 20640 rows and 10 columns


INFO: Missing values by column:
total_bedrooms    207


INFO: Removed 0 duplicate rows


INFO: Removed 0 invalid/non-physical rows


INFO: Removed 965 top-coded records at/above $500001


INFO: Removed 1100 outlier rows based on IQR rule for ['rooms_per_household', 'population_per_household']


INFO: Final cleaned shape: 18575 rows, 13 columns


INFO: Saved cleaned data to <project>/data/housing_clean.csv


## 2. Exploratory data analysis (`src/eda.py`)
Summary statistics and the figures saved to `figures/`.

In [3]:
%run src/eda.py

INFO: Loaded cleaned data: 18575 rows, 13 columns


INFO: Summary statistics:
                            count       mean       std       min        25%        50%        75%        max  skew
median_house_value        18575.0  193935.08  97841.88  14999.00  117100.00  176400.00  250900.00  500000.00  0.78
median_income             18575.0       3.69      1.54      0.50       2.55       3.48       4.61      13.15  0.80
housing_median_age        18575.0      28.53     12.49      1.00      18.00      29.00      37.00      52.00  0.07
rooms_per_household       18575.0       5.21      1.10      2.08       4.44       5.19       5.94       8.30  0.12
bedrooms_per_room         18575.0       0.21      0.06      0.04       0.18       0.20       0.24       1.00  2.49
population_per_household  18575.0       2.87      0.63      1.16       2.44       2.82       3.25       4.59  0.35


INFO: Saved all EDA figures to figures


## 3. Model selection (`src/model.py`)
Linear regression vs. random forest with 5-fold cross-validation; the better model is evaluated on the held-out test set and saved.

In [4]:
%run src/model.py

INFO: Loaded data for modeling: 18575 rows


INFO: Train size: 14860, Test size: 3715


INFO: Cross-validating linear_regression ...


INFO: linear_regression -> {'cv_rmse_mean': np.float64(57477.80084024812), 'cv_rmse_std': np.float64(1208.1145641161102), 'cv_rmse_folds': [57036.34929080748, 56203.2496761964, 56754.42955725143, 58187.45632015736, 59207.51935682797], 'rmse': np.float64(58464.119739235415), 'mae': 43136.71415892046, 'r2': 0.6515927047554535, 'train_rmse': np.float64(57433.13815154557), 'train_mae': 42646.60470833941, 'train_r2': 0.6532291965826214}


INFO: Cross-validating random_forest ...


INFO: random_forest -> {'cv_rmse_mean': np.float64(45203.5992903974), 'cv_rmse_std': np.float64(1391.495015087088), 'cv_rmse_folds': [44876.67846799184, 43565.04149295588, 45122.723864986394, 45032.42264792914, 47421.12997812373], 'rmse': np.float64(44715.158571241605), 'mae': 29899.237387539808, 'r2': 0.796193533159184, 'train_rmse': np.float64(24831.74643006597), 'train_mae': 15994.539378451145, 'train_r2': 0.9351765769423134}


INFO: Selected final model: random_forest


INFO: Saved model and metrics summary to <project>/models and <project>/reports


INFO: Top features:
                 feature  importance
           median_income    0.428546
  ocean_proximity_INLAND    0.180714
population_per_household    0.111351
               longitude    0.077372
                latitude    0.069072
      housing_median_age    0.041780
     rooms_per_household    0.024787
       bedrooms_per_room    0.022412
             total_rooms    0.010836
              population    0.009878


## 4. Model analysis (`src/rigor.py`)
Train/test gap, paired folds, bootstrap confidence intervals, multicollinearity (VIF), permutation importance, error by price and region, and spatial cross-validation.

In [5]:
%run src/rigor.py

TRAIN vs TEST {
 "linear": {
  "train_rmse": 57433.13815154557,
  "test_rmse": 58464.119739235415,
  "train_r2": 0.6532291965826214,
  "test_r2": 0.6515927047554535,
  "train_mae": 42646.60470833941,
  "test_mae": 43136.71415892046
 },
 "forest": {
  "train_rmse": 24831.746430065967,
  "test_rmse": 44715.158571241605,
  "train_r2": 0.9351765769423134,
  "test_r2": 0.796193533159184,
  "train_mae": 15994.539378451145,
  "test_mae": 29899.237387539808
 }
}


PAIRED FOLDS
  fold  lr_rmse  rf_rmse    diff
    1  57036.0  44877.0 12160.0
    2  56203.0  43565.0 12638.0
    3  56754.0  45123.0 11632.0
    4  58187.0  45032.0 13155.0
    5  59208.0  47421.0 11786.0
mean diff 12274  sd 627  95% CI (11496, 13053)  t=43.78 p=1.628e-06


BOOTSTRAP 95% CI RF test: RMSE [42752.92847772888, 46809.21732547043] R2 [0.7784668719185027, 0.8117285554015089] MAE [28820.516804447503, 31018.86124838968]
VIF {
 "households": 92.85565163189489,
 "total_bedrooms": 65.9309291898919,
 "population": 26.14733758296487,
 "total_rooms": 23.898151440428013,
 "latitude": 10.210663845143738,
 "longitude": 9.66174873316559,
 "rooms_per_household": 5.929642391047449,
 "bedrooms_per_room": 4.222679159521616,
 "population_per_household": 3.069234611185472,
 "median_income": 2.535218101923161,
 "housing_median_age": 1.3069316780797693
}
LR COEF (numeric = per 1 SD)
 ocean_proximity_ISLAND        122803.0
median_income                  60845.0
latitude                      -57227.0
longitude                     -54938.0
ocean_proximity_INLAND        -51668.0
ocean_proximity_NEAR BAY      -30891.0
households                     27740.0
ocean_proximity_NEAR OCEAN    -22093.0
population_per_household      -22068.0
ocean_proximity_<1H OCEAN     -18151

PERMUTATION IMPORTANCE (RMSE increase, $)
 median_income               42430.0
latitude                    20576.0
ocean_proximity             19897.0
longitude                   19357.0
population_per_household    17601.0
housing_median_age           6562.0
rooms_per_household          1172.0
bedrooms_per_room             716.0
total_rooms                   294.0
total_bedrooms                250.0
households                    129.0
population                    111.0
ERROR BY VALUE BIN
                            n      rmse      mae  bias_mean_pred_minus_actual
bin                                                                         
(0.0, 100000.0]        702.0   28077.0  17926.0                      15287.0
(100000.0, 200000.0]  1484.0   34988.0  23896.0                      13994.0
(200000.0, 300000.0]   967.0   42086.0  31622.0                       -325.0
(300000.0, 400000.0]   407.0   56191.0  42753.0                     -25724.0
(400000.0, 500000.0]   155.0  115286.0  971

SPATIAL CV  random 44347 ± 1393   spatial-block 59977 ± 7806
saved


## 5. Sensitivity to cleaning choices (`src/sensitivity.py`)

In [6]:
%run src/sensitivity.py

INFO: Loading raw data from <project>/data/housing.csv


INFO: Loaded 20640 rows and 10 columns


INFO: Removed 0 duplicate rows


INFO: Removed 0 invalid/non-physical rows


INFO: Removed 965 top-coded records at/above $500001


...

INFO: Loaded 20640 rows and 10 columns


INFO: Removed 0 duplicate rows


INFO: Removed 0 invalid/non-physical rows


INFO: Removed 965 top-coded records at/above $500001


                                                    variant  n_train  n_test   rf_rmse  rf_r2   lr_rmse  lr_r2  rf_rmse_under500k  lr_rmse_under500k
                                   baseline (team pipeline)    14860    3715 44715.159  0.796 58464.120  0.652                NaN                NaN
                                     no IQR outlier removal    15740    3935 46362.574  0.784 62105.213  0.613                NaN                NaN
                                  keep $500,001 capped rows    15544    3886 49500.614  0.819 64901.446  0.688          44990.176          57384.901
leakage-free (split first; train-only medians & IQR bounds)    14868    3711 46331.803  0.785 59520.567  0.645                NaN                NaN
                          leakage-free, test keeps outliers    14868    3935 46388.562  0.784 62177.376  0.612                NaN                NaN


## 6. Hyperparameter comparison (`src/hyperparam_check.py`)
Compares the 150-tree constrained forest with a 300-tree unconstrained forest.

In [7]:
%run src/hyperparam_check.py .

team (150 trees, depth 18, leaf 3): RMSE 44,715  R2 0.796  size 52.1 MB


larger (300 trees, unconstrained): RMSE 44,520  R2 0.798  size 402.3 MB
